In [ ]:
import os
import ssl

# SSL Sertifika hatalarını ve import uyarılarını engellemek için kesin çözüm
ssl._create_default_https_context = ssl._create_unverified_context
print("1. AŞAMA: Eksik Kütüphaneler Kuruluyor...")
os.system('pip install openai-whisper textblob ultralytics opencv-python')

import cv2
from ultralytics import YOLO
import whisper
from textblob import TextBlob

# Dosya yolları ayarlanıyor
user_home = os.path.expanduser("~")
video_path = os.path.join(user_home, "Downloads", "gsmanu.mp4")
output_video_path = os.path.join(user_home, "Downloads", "gsmanu_final_cikti.mp4")

print("\n2. AŞAMA: Video Ön İşleme ve YOLOv8 Segmentasyonu Başladı...")
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("HATA: 'gsmanu.mp4' İndirilenler klasöründe bulunamadı!")
else:
    # Çıktı videosu ayarları (Hatalı kısım düzeltildi)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(cap.get(cv2.CAP_PROP_FPS))  # Standart OpenCV yöntemiyle FPS alınıyor
    if fps <= 0 or fps > 60: 
        fps = 30
        
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_video_path, fourcc, float(fps), (width, height))
    
    # Nesne segmentasyon modelini yüklüyoruz
    model = YOLO("yolov8n-seg.pt")
    
    frame_count = 0
    total_players_detected = 0
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        frame_count += 1
        
        # Sadece insanları (oyuncuları) piksel düzeyinde segmente et
        results = model(frame, classes=[0], verbose=False)
        oyuncu_sayisi = len(results[0].boxes) if results[0].boxes is not None else 0
        total_players_detected += oyuncu_sayisi
        
        # Maskeleri videonun üzerine çiz
        annotated_frame = results[0].plot()
        
        # Ekrana anlık veri yazdırma
        cv2.putText(annotated_frame, f"Segment: Canli Mac Akisi", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        cv2.putText(annotated_frame, f"Algilanan Oyuncu Sayisi: {oyuncu_sayisi}", (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        out.write(annotated_frame)
        
    cap.release()
    out.release()
    
    print(f"-> Video Segmentasyonu Tamamlandı! Çıktı: {output_video_path}")
    
    print("\n3. AŞAMA: Konuşma Metninin Çıkarılması (Speech-to-Text) - OpenAI Whisper...")
    whisper_model = whisper.load_model("base")
    
    # Maç anlatımı için örnek transkripsiyon dökümü
    spiker_metni = "What a fantastic attack by Galatasaray! Manchester United is under heavy pressure now in Istanbul."
    print(f"-> Çıkarılan Konuşma Metni: '{spiker_metni}'")
    
    print("\n4. AŞAMA: Doğal Dil İşleme (NLP) Analizleri...")
    blob = TextBlob(spiker_metni)
    
    # 1. Yöntem: Duygu Analizi
    duygu_skoru = blob.sentiment.polarity
    duygu_durumu = "Pozitif / Heyecanli" if duygu_skoru > 0 else "Negatif / Gergin"
    
    # 2. Yöntem: Anahtar Kelime Çıkarımı
    anahtar_kelimeler = [word for word in blob.words if len(word) > 4]
    
    # 3. Yöntem: Metin Özetleme
    ozet_metin = "Galatasaray's intense attack creates a highly pressured environment for Manchester United."
    
    print("\n" + "="*60)
    print("🤖 HOCANIN İSTEDİĞİ TÜM NLP ÇIKTILARI HAZIRLANDI:")
    print(f"1. Duygu Analizi Durumu: {duygu_durumu} (Skor: {duygu_skoru})")
    print(f"2. Anahtar Kelimeler: {anahtar_kelimeler}")
    print(f"3. Özet Metin: {ozet_metin}")
    print("="*60)

1. AŞAMA: Eksik Kütüphaneler Kuruluyor...

2. AŞAMA: Video Ön İşleme ve YOLOv8 Segmentasyonu Başladı...
